# YOLO11 Fine-Tuning on VisDrone
Fine-tunes YOLO11s on VisDrone-DET and saves best weights as a Kaggle output.

In [ ]:
# 1. Check GPU - verify CUDA is available and print GPU name and VRAM
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Install packages - install Ultralytics YOLO and supporting libraries
!pip install ultralytics pillow pyyaml pycocotools -q

In [ ]:
# 4. Imports - load all Python libraries used across this notebook
import os, json, re, time
from pathlib import Path
import torch


In [ ]:
# 5. Locate dataset - print available paths so you can confirm the dataset location
import os

def explore(path, depth=0, max_depth=4):
    if depth > max_depth:
        return
    try:
        for item in sorted(os.listdir(path)):
            print('  ' * depth + item)
            full = os.path.join(path, item)
            if os.path.isdir(full):
                explore(full, depth + 1, max_depth)
    except PermissionError:
        pass

explore('/kaggle/input')

In [ ]:
# 6. Set dataset path - update VISDRONE_ROOT to match the path printed above
# It should point to the folder that contains VisDrone2019-DET-train and VisDrone2019-DET-val
VISDRONE_ROOT = '/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset'

In [ ]:
# 7. Write dataset YAML - generate the config file YOLO needs for training
import yaml
from pathlib import Path

YOLO_CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]

dataset_yaml = {
    'train': f'{VISDRONE_ROOT}/VisDrone2019-DET-train/images',
    'val':   f'{VISDRONE_ROOT}/VisDrone2019-DET-val/images',
    'names': {i: n for i, n in enumerate(YOLO_CLASS_NAMES)},  # YOLO expects a zero-indexed {id: name} dict
}

yaml_path = Path('/kaggle/working/dataset.yaml')
yaml_path.write_text(yaml.dump(dataset_yaml))
print('dataset.yaml written')
print(yaml_path.read_text())

In [ ]:
# 8. Train YOLO11s - fine-tune on VisDrone for the configured number of epochs
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data     = '/kaggle/working/dataset.yaml',
    epochs   = 50,
    imgsz    = 640,
    batch    = 16,
    device   = 0,
    project  = '/kaggle/working/models',
    name     = 'yolo11s_visdrone',
    patience = 15,        # stop early if val mAP hasn't improved for 15 epochs
    cache    = False,
    mosaic   = 1.0,
    close_mosaic = 10,   # disable mosaic augmentation for the final 10 epochs to stabilize
    cos_lr   = True,
    plots    = True,
    save     = True,
)

print('\nTraining complete.')
print('Best weights saved to:', results.save_dir, '/ weights/best.pt')

In [ ]:
# 9. Verify weights - confirm best.pt was saved and print its location
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
print('best.pt exists:', best_pt.exists())
print('Size:', round(best_pt.stat().st_size / 1e6, 1), 'MB')
print('\nDownload best.pt from the Output tab on the right ->')